# Will this Patient be Readmitted Within 30 Days?

**Input:**
- Discharge Summary
- Prior Readmissions
- Diagnosis

**Output:** Yes/No

## Possible Models:
- Logistic Regression
- Random Forest
- Gradient Boosted Trees

In [ ]:
import numpy as np
import pandas as pd
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from pyspark.sql.functions import col, to_date, datediff, unix_timestamp, lead, when, year
from pyspark.sql.types import IntegerType
from pyspark.sql import SparkSession
from pyspark.sql import Window
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, precision_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
import requests
import json
from urllib.parse import quote
import logging
from datetime import datetime
import matplotlib.pyplot as plt

# Get the Dataset

In [ ]:


# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(message)s',
    handlers=[
        logging.FileHandler(f'synthea_load_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'),
        logging.StreamHandler()
    ]
)

def load_synthea_data(config_path=None, config_url=None, error_log_path="error_log.txt"):
    # Load config from local file or URL
    if config_url:
        config_response = requests.get(config_url)
        config = config_response.json()
    elif config_path:
        with open(config_path, 'r') as f:
            config = json.load(f)
    else:
        raise ValueError("Must provide either config_path or config_url")
    
    base_url = config['base_url']
    files = config['files']
    
    data = {}
    for file in files:
        try:
            encoded_file = quote(file)
            url = f"{base_url}/{encoded_file}"
            df = pd.read_csv(url)
            key = file.replace('.csv', '')
            data[key] = df
            print(f"Loaded {file}: {len(df)} rows")
        except Exception as e:
            logging.error(f"Error loading {file}: {e}")

    
    return data


# Load from GitHub-hosted JSON config
config_url = "https://raw.githubusercontent.com/Nicholas26-design/PorfolioProjects/main/Synthea/synthea_config"
dataframes = load_synthea_data(config_url=config_url)

# Cleaning

In [ ]:
# Convert Pandas DataFrames to Spark DataFrames
spark = SparkSession.builder.appName("Synthea").getOrCreate()

spark_dataframes = {table_name: spark.createDataFrame(df) for table_name, df in dataframes.items()}

# Example: Access Spark DataFrames like spark_dataframes["condition_"], spark_dataframes["patient_"], etc.

In [0]:
# Load the encounter_ and patient_ dataframes from the dictionary
encounter_pandas_df = dataframes["encounter_"]

# Convert the Pandas DataFrame to Spark DataFrame
encounter_spark_df = spark.createDataFrame(encounter_pandas_df)

# Convert ISO 8601 string to datetime using Spark functions
encounter_spark_df = (
    encounter_spark_df.withColumn("start_date", to_date(col("start_time")))
    .withColumn("end_date", to_date(col("end_time")))
    .withColumn("duration_days", datediff(col("end_date"), col("start_date")))
)

patient_df = dataframes["patient_"]
patient_df["first_name"] = patient_df["first_name"].str.replace(r"\d+", "", regex=True)
patient_df["last_name"] = patient_df["last_name"].str.replace(r"\d+", "", regex=True)

# Convert the Pandas DataFrame back to Spark DataFrame
patient_spark_df = spark.createDataFrame(patient_df)

patient_spark_df = patient_spark_df.withColumn("birth_date", to_date(col("birth_date")))

# Perform the left outer join
patient_encounters_df = encounter_spark_df.join(
    patient_spark_df, on="patient_id", how="left_outer"
)

# Display the result
display(patient_encounters_df)

# Exploratory Analysis

In [0]:
# Count the number of encounters per patient
encounter_count_df = patient_encounters_df.groupBy(
    "patient_id", "first_name", "last_name"
).count()
display(encounter_count_df)

In [0]:
# Sort by patient_id and start_time using a window
w = Window.partitionBy("patient_id").orderBy("start_time")

# Add next encounter's start_time
df_sorted = patient_encounters_df.withColumn(
    "next_start_time", lead("start_time").over(w)
)

# Calculate days until next encounter
df_sorted = df_sorted.withColumn(
    "days_until_next", datediff(col("next_start_time"), col("end_time"))
)

# Create readmission label: 1 if within 30 days, 0 otherwise
df_sorted = df_sorted.withColumn(
    "readmitted_within_30_days",
    when((col("days_until_next") >= 0) & (col("days_until_next") <= 30), 1).otherwise(
        0
    ),
)

# Optional: check distribution
df_sorted.groupBy("readmitted_within_30_days").count().show()

# Feature and Target Selection

In [0]:
# Calculate age: year(start_time) - year(birth_date)
df_clean = df_sorted.withColumn("age", year("start_time") - year("birth_date"))

# Define feature columns again
features_to_keep = [
    "status",
    "encounter_type_code",
    "encounter_type_text",
    "duration_days",
    "gender",
    "marital_status",
    "age",
]

# Define target column
target_col = "readmitted_within_30_days"

# Select relevant columns from the PySpark DataFrame
selected_cols = features_to_keep + [target_col]
df_pandas = df_clean.select(*selected_cols).toPandas()

# Drop rows with missing values
df_pandas = df_pandas.dropna()

# Split into features (X) and target (y)
X = df_pandas[features_to_keep]
y = df_pandas[target_col]

# Preprocessing

In [0]:
# Define categorical and numeric columns
categorical_features = ["status", "encounter_type_text", "gender", "marital_status"]
numeric_features = ["encounter_type_code", "duration_days", "age"]

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="mean")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        ),
        (
            "cat",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_features,
        ),
    ]
)

# Now X and y are ready to be used in a model pipeline

# Modeling

In [0]:
# set the experiment id
mlflow.set_experiment(experiment_id="336962172052827")

mlflow.autolog()

# --- Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- Full modeling pipeline ---
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, solver="lbfgs")),
    ]
)

# --- Train the model ---
model.fit(X_train, y_train)

# --- Evaluate ---
# By default, models predict Class 1 if the probability is greater than or equal to 0.5.

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

# Calibration

In [0]:
# Get probabilities for the positive class (Class 1)
y_probs = model.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.1, 0.9, 0.05)
prec_class_0 = []
prec_class_1 = []

# Compute precision for both classes at different thresholds
for t in thresholds:
    y_pred_thresh = (y_probs >= t).astype(int)
    prec_class_0.append(precision_score(y_test, y_pred_thresh, pos_label=0))
    prec_class_1.append(precision_score(y_test, y_pred_thresh, pos_label=1))

# Plot both
plt.figure(figsize=(8, 6))
plt.plot(thresholds, prec_class_0, marker='o', label='Class 0 Precision', color='blue')
plt.plot(thresholds, prec_class_1, marker='s', label='Class 1 Precision', color='green')
plt.xlabel("Threshold for Predicting Class 1")
plt.ylabel("Precision")
plt.title("Precision vs. Threshold for Both Classes")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## Second Run
0.5714285714285714: Class 0 Precision (This is the most precision you get)
0.9147286821705426: Class 1 Precision (This is the precision you get when class 0 is at its most precise)
Threshold at these points: 0.65

In [0]:
# Set the experiment id
mlflow.set_experiment(experiment_id="336962172052827")

mlflow.autolog()

# --- Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- Full modeling pipeline ---
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, solver="lbfgs")),
    ]
)

# --- Train the model ---
model.fit(X_train, y_train)

# --- Evaluate with specified threshold ---
y_probs = model.predict_proba(X_test)[:, 1]
threshold = 0.65
y_pred_thresh = (y_probs >= threshold).astype(int)

print(classification_report(y_test, y_pred_thresh))